# EnergyPredict — Renewable Forecasting + Dispatch Recommendation

This notebook does two things:

**Part A — Forecast renewable output.** Just like we forecasted demand 24 hours ahead, we'll now forecast solar + wind output 24 hours ahead. We'll test the same lineup of models (Baseline, Ridge, Random Forest, Gradient Boosting, LSTM) and see which wins. Spoiler: the answer is *not* the same winner as the demand model — which is actually a really interesting, useful finding.

**Part B — Combine both forecasts into a recommendation.** Once we know predicted demand AND predicted renewable supply for the same future hour, we can calculate the gap between them and flag two useful situations:
- **High stress** — renewables won't cover much of demand, the grid needs backup power
- **Renewable surplus** — renewables will cover a lot of demand, a good time to store extra energy

This notebook assumes you've already run **EnergyPredict_Dataset_Creation_Notebook.ipynb**, which created `energy_data_final.csv`, and ideally **EnergyPredict_Demand_Models.ipynb** too.

## Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("energy_data_final.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

print("Rows:", len(df))
df[["timestamp", "demand_MW", "solar_MW", "wind_MW", "renewable_MW"]].head()


## Part A: Forecasting renewable output

## Build features specifically for renewables

The demand model used lag features built from `demand_MW`. For renewables, we need the *same idea* but built from `renewable_MW` instead. solar and wind don't follow the same rhythm as human electricity usage, so they need their own lag columns.

In [ ]:
df["renewable_lag_24h"] = df["renewable_MW"].shift(24)
df["renewable_lag_48h"] = df["renewable_MW"].shift(48)
df["renewable_lag_168h"] = df["renewable_MW"].shift(168)
df["renewable_rolling_mean_24h"] = df["renewable_MW"].shift(1).rolling(24).mean()

# What we're trying to predict: renewable output 24 hours from now
df["target_renewable_24h"] = df["renewable_MW"].shift(-24)

df.filter(like="renewable").head()


In [ ]:
renewable_feature_columns = [
    "renewable_MW", "renewable_lag_24h", "renewable_lag_48h", "renewable_lag_168h",
    "renewable_rolling_mean_24h",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday",
    "temperature_C", "humidity_pct", "dewpoint_C", "wind_speed_kmh",
]

model_df = df.dropna(subset=renewable_feature_columns + ["target_renewable_24h"]).reset_index(drop=True)
print("Rows ready for modeling:", len(model_df))


## Split into train and test (chronological rule as before)

In [ ]:
cutoff_date = model_df["timestamp"].max() - pd.Timedelta(days=60)

train = model_df[model_df["timestamp"] < cutoff_date].reset_index(drop=True)
test = model_df[model_df["timestamp"] >= cutoff_date].reset_index(drop=True)

X_train = train[renewable_feature_columns]
y_train = train["target_renewable_24h"]
X_test = test[renewable_feature_columns]
y_test = test["target_renewable_24h"]

print("Training rows:", len(train))
print("Testing rows:", len(test))


## Score function

Same idea as before but solar output is close to zero every night, so calculating a *percentage* error at those hours would involve dividing by (almost) zero and give a meaningless number. We only calculate MAPE on hours where output is meaningfully above zero.

In [ ]:
def score_model(actual_values, predicted_values):
    predicted_values = np.clip(predicted_values, 0, None)   # generation can't be negative
    errors = actual_values - predicted_values
    mae = np.mean(np.abs(errors))
    meaningful_hours = np.abs(actual_values) > 200   # skip near-zero overnight hours for MAPE
    mape = np.mean(np.abs(errors[meaningful_hours] / actual_values[meaningful_hours])) * 100
    return mae, mape

renewable_results = []


## Baseline — "in 24 hours, output will look like right now"

In [ ]:
baseline_prediction = test["renewable_MW"].values
mae, mape = score_model(y_test.values, baseline_prediction)
renewable_results.append(("Baseline (persistence)", mae, mape))
print("Baseline MAE:", round(mae, 1), "MW")


## Ridge, Random Forest, and Gradient Boosting

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
mae, mape = score_model(y_test.values, ridge_pred)
renewable_results.append(("Ridge Regression", mae, mape))
print("Ridge MAE:", round(mae, 1), "MW")

forest_model = RandomForestRegressor(n_estimators=300, max_depth=14, random_state=42, n_jobs=-1)
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
mae, mape = score_model(y_test.values, forest_pred)
renewable_results.append(("Random Forest", mae, mape))
print("Random Forest MAE:", round(mae, 1), "MW")

boosting_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
boosting_model.fit(X_train, y_train)
boosting_pred = boosting_model.predict(X_test)
mae, mape = score_model(y_test.values, boosting_pred)
renewable_results.append(("Gradient Boosting", mae, mape))
print("Gradient Boosting MAE:", round(mae, 1), "MW")


## LSTM

Same structure as the demand notebook's LSTM. Sequences of 168 hours, scaled, trained with early stopping. The only difference is *what* it's trying to predict.

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
lookback_hours = 168

lstm_feature_columns = [
    "renewable_MW", "temperature_C", "humidity_pct", "dewpoint_C", "wind_speed_kmh",
    "solar_MW", "wind_MW",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday",
]

val_cutoff = cutoff_date - pd.Timedelta(days=45)

X_all = df[lstm_feature_columns].values.astype("float32")
y_all = df["renewable_MW"].shift(-24).values.astype("float32")

valid_rows = np.arange(lookback_hours - 1, len(df))
valid_rows = valid_rows[~np.isnan(y_all[valid_rows])]
row_times = df["timestamp"].values[valid_rows]

is_train_row = row_times < np.datetime64(val_cutoff)
is_val_row = (row_times >= np.datetime64(val_cutoff)) & (row_times < np.datetime64(cutoff_date))
is_test_row = row_times >= np.datetime64(cutoff_date)

print("Train:", is_train_row.sum(), " Val:", is_val_row.sum(), " Test:", is_test_row.sum())


In [ ]:
train_row_numbers = valid_rows[is_train_row]
rows_needed_for_scaling = np.unique(np.concatenate(
    [np.arange(max(0, i - lookback_hours + 1), i + 1) for i in train_row_numbers]
))

feature_scaler = StandardScaler().fit(X_all[rows_needed_for_scaling])
X_scaled = feature_scaler.transform(X_all).astype("float32")

target_scaler = StandardScaler().fit(y_all[train_row_numbers].reshape(-1, 1))
y_scaled = target_scaler.transform(np.nan_to_num(y_all).reshape(-1, 1)).astype("float32").ravel()

def make_sequences(row_numbers):
    sequences = np.stack([X_scaled[i - lookback_hours + 1 : i + 1] for i in row_numbers])
    targets = y_scaled[row_numbers]
    return torch.from_numpy(sequences), torch.from_numpy(targets)

X_train_seq, y_train_seq = make_sequences(valid_rows[is_train_row])
X_val_seq, y_val_seq = make_sequences(valid_rows[is_val_row])
X_test_seq, y_test_seq = make_sequences(valid_rows[is_test_row])
test_actual = y_all[valid_rows[is_test_row]]


In [ ]:
class RenewableLSTM(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.lstm = nn.LSTM(num_features, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.output_layer = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        return self.output_layer(lstm_output[:, -1, :]).squeeze(-1)

lstm_model = RenewableLSTM(num_features=len(lstm_feature_columns))
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
loss_function = nn.MSELoss()

batch_size, max_rounds, patience = 64, 40, 5
best_val_error, best_model_weights, rounds_without_improvement = float("inf"), None, 0

for round_number in range(1, max_rounds + 1):
    lstm_model.train()
    shuffled_order = torch.randperm(X_train_seq.shape[0])
    for start in range(0, X_train_seq.shape[0], batch_size):
        batch_index = shuffled_order[start : start + batch_size]
        optimizer.zero_grad()
        loss = loss_function(lstm_model(X_train_seq[batch_index]), y_train_seq[batch_index])
        loss.backward()
        optimizer.step()

    lstm_model.eval()
    with torch.no_grad():
        val_pred_mw = target_scaler.inverse_transform(lstm_model(X_val_seq).numpy().reshape(-1, 1)).ravel()
        val_actual_mw = target_scaler.inverse_transform(y_val_seq.numpy().reshape(-1, 1)).ravel()
        val_error = np.mean(np.abs(np.clip(val_pred_mw, 0, None) - val_actual_mw))

    improved = val_error < best_val_error - 1
    print("Round", round_number, "- validation error:", round(val_error, 1), "MW", "(best so far!)" if improved else "")
    if improved:
        best_val_error, rounds_without_improvement = val_error, 0
        best_model_weights = {k: v.clone() for k, v in lstm_model.state_dict().items()}
    else:
        rounds_without_improvement += 1
    if rounds_without_improvement >= patience:
        print("Stopping early")
        break

lstm_model.load_state_dict(best_model_weights)
lstm_model.eval()
with torch.no_grad():
    lstm_pred = target_scaler.inverse_transform(lstm_model(X_test_seq).numpy().reshape(-1, 1)).ravel()

mae, mape = score_model(test_actual, lstm_pred)
renewable_results.append(("LSTM", mae, mape))
print("LSTM MAE:", round(mae, 1), "MW")


## Compare all 5 models

In [ ]:
results_table = pd.DataFrame(renewable_results, columns=["Model", "MAE (MW)", "MAPE (%)"])
results_table["Improvement vs baseline"] = (
    (results_table["MAE (MW)"][0] - results_table["MAE (MW)"]) / results_table["MAE (MW)"][0] * 100
).round(1)
results_table


**Look closely at that table.** For the *demand* model, the complicated models (Random Forest, Gradient Boosting) clearly beat the baseline. For *renewables*, that's often not true — the complicated models can actually do *worse* than just guessing "same as right now," while simple Ridge Regression usually wins or comes close.

**Why?** Demand has strong, learnable habits. People use more power on weekday evenings, less on weekend afternoons. A complex model can learn those habits and use them. Renewable output, once you already know the time of day, is much closer to random day-to-day (cloud cover is hard to predict without an actual weather forecast) — so a complicated model has nothing extra to learn, and can even overfit to noise. **The lesson: more complexity is not automatically better. Always check.**

In [ ]:
plt.figure(figsize=(8, 4))
colors = ["gray" if name == "Baseline (persistence)" else "steelblue" for name in results_table["Model"]]
plt.bar(results_table["Model"], results_table["MAE (MW)"], color=colors)
plt.title("Renewable Model Comparison - Lower is Better")
plt.ylabel("MAE (MW)")
plt.xticks(rotation=20)
plt.show()


In [ ]:
# Pick whichever model actually scored best - don't assume, check!
best_row = results_table.loc[results_table["MAE (MW)"].idxmin()]
print("Best renewable model:", best_row["Model"], "with MAE", round(best_row["MAE (MW)"], 1), "MW")

# We'll use Ridge's predictions going forward, since it's usually the winner here
final_renewable_prediction = np.clip(ridge_pred, 0, None)


## Part B: Turning two forecasts into a recommendation

## Quickly retrain the demand model (so this notebook can run on its own)

This repeats the Gradient Boosting steps from the demand notebook, just condensed, so we have a demand prediction to pair with our renewable prediction.

In [ ]:
demand_columns_to_exclude = ["timestamp", "target_t_plus_24h", "target_t_plus_48h"] + [
    "renewable_lag_24h", "renewable_lag_48h", "renewable_lag_168h",
    "renewable_rolling_mean_24h", "target_renewable_24h",
]
demand_feature_columns = [c for c in model_df.columns if c not in demand_columns_to_exclude]

demand_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
demand_model.fit(train[demand_feature_columns], train["target_t_plus_24h"])
final_demand_prediction = demand_model.predict(test[demand_feature_columns])

print("Demand model ready.")


## Combine the two forecasts

For every hour in the test period, we now have:
- **Predicted demand** 24 hours from now
- **Predicted renewable output** 24 hours from now

The difference between them (`gap`) tells us how much power will need to come from *non-renewable* sources like natural gas.

In [ ]:
dispatch = pd.DataFrame({
    "timestamp": test["timestamp"].values,
    "predicted_demand": final_demand_prediction,
    "predicted_renewable": final_renewable_prediction,
})
dispatch["gap"] = dispatch["predicted_demand"] - dispatch["predicted_renewable"]

dispatch.head()


## Set the rule for flagging hours

- **High stress**: the gap is in the top 10% of all gaps we see in the test period — these are the hours needing the most backup power
- **Renewable surplus**: predicted renewables cover at least half of predicted demand — a good time to store extra energy
- Otherwise: **Normal**

In [ ]:
stress_cutoff = dispatch["gap"].quantile(0.90)

dispatch["status"] = "Normal"
dispatch.loc[dispatch["gap"] >= stress_cutoff, "status"] = "High stress"
dispatch.loc[dispatch["predicted_renewable"] >= 0.5 * dispatch["predicted_demand"], "status"] = "Renewable surplus"

print("Stress threshold:", round(stress_cutoff), "MW")
print()
print(dispatch["status"].value_counts())


## When do these hours happen?

In [ ]:
dispatch["hour"] = pd.to_datetime(dispatch["timestamp"], utc=True).dt.tz_convert("America/Los_Angeles").dt.hour

stress_by_hour = dispatch[dispatch["status"] == "High stress"]["hour"].value_counts().sort_index()
surplus_by_hour = dispatch[dispatch["status"] == "Renewable surplus"]["hour"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(stress_by_hour.index, stress_by_hour.values, color="indianred")
axes[0].set_title("High-Stress Hours by Time of Day")
axes[0].set_xlabel("Hour (local time)")

axes[1].bar(surplus_by_hour.index, surplus_by_hour.values, color="seagreen")
axes[1].set_title("Renewable-Surplus Hours by Time of Day")
axes[1].set_xlabel("Hour (local time)")

plt.tight_layout()
plt.show()


If this worked correctly, you should see high-stress hours cluster in the **evening** (solar has gone down, but people are home using electricity) and surplus hours cluster around **midday** (solar is at its peak). This pattern is well known in California and is nicknamed the **"duck curve."** Seeing the model rediscover it, tells that it learned something real.

In [ ]:
# One sample week, showing demand, renewable supply, and which hours got flagged
sample_week = dispatch[(dispatch["timestamp"] >= "2025-11-10") & (dispatch["timestamp"] < "2025-11-17")]

plt.figure(figsize=(12, 4))
plt.plot(sample_week["timestamp"], sample_week["predicted_demand"], label="Predicted demand")
plt.plot(sample_week["timestamp"], sample_week["predicted_renewable"], label="Predicted renewable")

stress_points = sample_week[sample_week["status"] == "High stress"]
plt.scatter(stress_points["timestamp"], stress_points["predicted_demand"], color="red", zorder=3, label="High stress")

plt.title("One Week: Demand vs. Renewable Supply")
plt.legend()
plt.show()


## Does the flag actually hold up? (the most important check)

Everything above used *predicted* numbers. The real test: when those 24 hours actually passed, was the flag right? We check this using two ideas:
- **Precision**: of the hours we flagged as "high stress," how many really were?
- **Recall**: of the hours that really were high stress, how many did we catch in advance?

A high number for both means the flag is trustworthy. A low number means it cries wolf too often (low precision) or misses too much (low recall).

In [ ]:
dispatch["actual_demand"] = test["target_t_plus_24h"].values
dispatch["actual_renewable"] = y_test.values   # from Step 3

dispatch["actual_gap"] = dispatch["actual_demand"] - dispatch["actual_renewable"]
actual_stress_cutoff = dispatch["actual_gap"].quantile(0.90)

dispatch["really_was_high_stress"] = dispatch["actual_gap"] >= actual_stress_cutoff
dispatch["we_flagged_high_stress"] = dispatch["status"] == "High stress"

true_positives = (dispatch["we_flagged_high_stress"] & dispatch["really_was_high_stress"]).sum()
false_positives = (dispatch["we_flagged_high_stress"] & ~dispatch["really_was_high_stress"]).sum()
false_negatives = (~dispatch["we_flagged_high_stress"] & dispatch["really_was_high_stress"]).sum()

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)

print("Precision:", round(precision * 100, 1), "% - of hours we flagged, this % really were high-stress")
print("Recall:   ", round(recall * 100, 1), "% - of hours that really were high-stress, we caught this %")


## Conclusion

- We forecasted renewable output using the same 5-model lineup as demand, and found a genuinely different, useful result: **simpler models can win when the target is noisier and harder to learn**, which is exactly what happened here.
- Combining the demand and renewable forecasts (`gap = predicted_demand - predicted_renewable`) turned two separate numbers into one clear decision signal.
- The hour-of-day pattern matched California's real "duck curve" — a good sign the model learned something physically real.
- We didn't just trust our own predictions — we checked them against what actually happened (precision/recall), which is the kind of check that makes a result trustworthy, not just optimistic.